In [ ]:
"""
Name: Module 6 Academic Success.py
Course: Data Preparation and Analysis
Created Date: November 4, 2023
Author: Ming-Long Lam, Ph.D.g
Organization: Illinois Institute of Technology
"""

In [ ]:
import matplotlib.pyplot as plt
import numpy
import pandas
import sys

In [ ]:
from scipy.stats import chi2

In [ ]:
sys.path.append('C:\\IIT\\Data Preparation and Analysis\\Code')
import Utility

In [ ]:
# Set some options for printing all the columns
numpy.set_printoptions(precision = 10, threshold = sys.maxsize)
numpy.set_printoptions(linewidth = numpy.inf)

In [ ]:
pandas.set_option('display.max_columns', None)
pandas.set_option('display.expand_frame_repr', False)
pandas.set_option('max_colwidth', None)

In [ ]:
pandas.options.display.float_format = '{:,.10f}'.format

In [ ]:
success = pandas.read_csv('C:\\IIT\\Data Preparation and Analysis\\Data\\Student Academic Success.csv', delimiter = ';')

In [ ]:
print(success.groupby('Target').size())

In [ ]:
success['Dropout'] = success['Target'].map({'Dropout': 'Yes', 'Enrolled': 'No', 'Graduate': 'No'})
q_dropout_count = success.groupby('Dropout').size()
q_dropout_odds = q_dropout_count['Yes'] / q_dropout_count['No']

In [ ]:
success['Attendance'] = success['Daytime/evening attendance'].map({0: 'Evening', 1: 'Daytime'})
success['Debtor'] = success['Debtor'].map({0: 'No', 1: 'Yes'})
success['Scholarship holder'] = success['Scholarship holder'].map({0: 'No', 1: 'Yes'})

In [ ]:
# Univariate frequency bar charts
fig, (ax1, ax2, ax3, ax4) = plt.subplots(nrows = 1, ncols = 4, dpi = 200, sharey = True, figsize = (16,6))
fig.tight_layout()

In [ ]:
ufreq = success['Dropout'].astype('category').value_counts()
ax1.bar(ufreq.index, ufreq, color = 'red')
ax1.set_xlabel('Drop Out College?')
ax1.set_ylabel('Number of Observations')
ax1.set_yticks(range(0, 4500, 500))
ax1.yaxis.grid(True)

In [ ]:
ufreq = success['Attendance'].astype('category').value_counts()
ax2.bar(ufreq.index, ufreq, color = 'dodgerblue')
ax2.set_xlabel('Mode of Attendance')
ax2.set_ylabel('')
ax2.yaxis.grid(True)

In [ ]:
ufreq = success['Debtor'].astype('category').value_counts()
ax3.bar(ufreq.index, ufreq, color = 'orange')
ax3.set_xlabel('Owe Debt?')
ax3.set_ylabel('')
ax3.yaxis.grid(True)

In [ ]:
ufreq = success['Scholarship holder'].astype('category').value_counts()
ax4.bar(ufreq.index, ufreq, color = 'green')
ax4.set_xlabel('Hold Scholarship?')
ax4.set_ylabel('')
ax4.yaxis.grid(True)

In [ ]:
plt.show()

In [ ]:
# Univariate odds of dropout
fig, (ax1, ax2, ax3) = plt.subplots(nrows = 1, ncols = 3, dpi = 200, sharey = True, figsize = (16,6))
fig.tight_layout()

In [ ]:
xtab = pandas.crosstab(success['Attendance'], success['Dropout'])
odds_table =  xtab['Yes'] / xtab['No']

In [ ]:
ax1.bar(odds_table.index, odds_table, color = 'dodgerblue')
ax1.axhline(y = q_dropout_odds, color = 'red', linestyle = '--')
ax1.set_xlabel('Mode of Attendance')
ax1.set_ylabel('Dropout Odds = Yes/No')
ax1.yaxis.grid(True)

In [ ]:
xtab = pandas.crosstab(success['Debtor'], success['Dropout'])
odds_table =  xtab['Yes'] / xtab['No']

In [ ]:
ax2.bar(odds_table.index, odds_table, color = 'orange')
ax2.axhline(y = q_dropout_odds, color = 'red', linestyle = '--')
ax2.set_xlabel('Owe Debt?')
ax2.set_ylabel('')
ax2.yaxis.grid(True)

In [ ]:
xtab = pandas.crosstab(success['Scholarship holder'], success['Dropout'])
odds_table =  xtab['Yes'] / xtab['No']

In [ ]:
ax3.bar(odds_table.index, odds_table, color = 'green')
ax3.axhline(y = q_dropout_odds, color = 'red', linestyle = '--')
ax3.set_xlabel('Hold Scholarship?')
ax3.set_ylabel('')
ax3.yaxis.grid(True)

In [ ]:
plt.show()

In [ ]:
xtab = pandas.crosstab(success['Age at enrollment'], success['Dropout'])
print(xtab)

In [ ]:
odds_table =  xtab['Yes'] / xtab['No']

In [ ]:
plt.figure(figsize = (10,4), dpi = 200)
plt.plot(odds_table.index, odds_table, marker = 'o', color = 'royalblue')
plt.axhline(y = q_dropout_odds, color = 'red', linestyle = '--')
plt.xlabel('Age of Enrollment')
plt.ylabel('Dropout Odds = Yes/No')
plt.xticks(numpy.arange(15,75,5))
plt.grid(axis = 'both')
plt.show()

In [ ]:
# Forward Selection
catName = ['Attendance', 'Debtor', 'Scholarship holder']
intName = ['Age at enrollment']
yName = 'Dropout'
yCat = ['No', 'Yes']

In [ ]:
train_data = success[catName + intName + [yName]]

In [ ]:
step_diary = []
y_train = train_data[yName]

In [ ]:
# Intercept only model
X0_train = train_data[[]].copy()
X0_train.insert(0, 'Intercept', 1.0)

In [ ]:
result_list = Utility.MNLogisticModel (X0_train, y_train)
n_iter = result_list[0].mle_retvals['iterations']
llk0 = result_list[1]
df0 = result_list[2]

In [ ]:
step_diary.append([0, 'Intercept', ' ', n_iter, df0, llk0, numpy.nan, numpy.nan, numpy.nan])

In [ ]:
entryThreshold = 0.05

In [ ]:
cName = catName.copy()
iName = intName.copy()
nPredictor = len(cName) + len(iName)

In [ ]:
# The Deviance significance is the eighth element in each row of the test result
def takeDevSig(s):
    return s[7]

In [ ]:
for step in range(nPredictor):
    enterName = ''

    # Columns are 'Predictor', 'Type', 'N Iter', 'ModelDF', 'ModelLLK', 'DevChiSq', 'DevDF', 'DevSig'
    step_detail = []

    # Enter the next predictor
    for X_name in cName:
        X_train = X0_train.join(pandas.get_dummies(train_data[[X_name]].astype('category'), dtype = float))
        result_list = Utility.MNLogisticModel (X_train, y_train)
        n_iter = result_list[0].mle_retvals['iterations']
        llk1 = result_list[1]
        df1 = result_list[2]
        devChiSq = 2.0 * (llk1 - llk0)
        devDF = df1 - df0
        devSig = chi2.sf(devChiSq, devDF)
        step_detail.append([X_name, 'categorical', n_iter, df1, llk1, devChiSq, devDF, devSig])

    for X_name in iName:
        X_train = X0_train.join(train_data[[X_name]])
        result_list = Utility.MNLogisticModel (X_train, y_train)
        n_iter = result_list[0].mle_retvals['iterations']
        llk1 = result_list[1]
        df1 = result_list[2]
        devChiSq = 2.0 * (llk1 - llk0)
        devDF = df1 - df0
        devSig = chi2.sf(devChiSq, devDF)
        step_detail.append([X_name, 'interval', n_iter, df1, llk1, devChiSq, devDF, devSig])

    # Find a predictor to add, if any
    step_detail.sort(key = takeDevSig, reverse = False)
    minSig = takeDevSig(step_detail[0])
    if (minSig <= entryThreshold):
        add_var = step_detail[0][0]
        add_type = step_detail[0][1]
        df0 = step_detail[0][3]
        llk0 = step_detail[0][4]
        step_diary.append([step+1] + step_detail[0])
        if (add_type == 'categorical'):
           X0_train = X0_train.join(pandas.get_dummies(train_data[[add_var]].astype('category'), dtype = float))
           cName.remove(add_var)
        else:
           X0_train = X0_train.join(train_data[[add_var]])
           iName.remove(add_var)           
    else:
        break

In [ ]:
# End of forward selection
print('\n======= Step Summary =======')
step_diary = pandas.DataFrame(step_diary, columns = ['Step', 'Predictor', 'Type', 'N Iter', 'ModelDF', \
                              'ModelLLK', 'DevChiSq', 'DevDF', 'DevSig'])
print(step_diary)

In [ ]:
# Retrain the final model
result_list = Utility.MNLogisticModel (X0_train, y_train)
thisFit = result_list[0]

In [ ]:
print(thisFit.summary())

In [ ]:
beta =  thisFit.params
aliasParam = result_list[4]
nonAliasParam = result_list[5]

In [ ]:
print('=== Aliased Columns in Model Matrix X ===')
print(X0_train.columns[aliasParam])

In [ ]:
llk1 = thisFit.llf
llk0 = thisFit.llnull
n_sample = X0_train.shape[0]

In [ ]:
R_MF = 1.0 - (llk1 / llk0)

In [ ]:
R_CS = (2.0 / n_sample) * (llk0 - llk1)
R_CS = 1.0 - numpy.exp(R_CS)

In [ ]:
upbound = (2.0 / n_sample) * llk0
upbound = 1.0 - numpy.exp(upbound)
R_N = R_CS / upbound

In [ ]:
predprob_event = thisFit.predict(X0_train.iloc[:, list(nonAliasParam)])[1]

In [ ]:
S1 = numpy.mean(predprob_event[y_train == 'Yes'])
S0 = numpy.mean(predprob_event[y_train == 'No'])

In [ ]:
R_TJ = S1 - S0